In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U "trl==1.5.1" "transformers>=4.56.2,<5.0" "peft>=0.14.0" "bitsandbytes>=0.43.0" "huggingface_hub>=0.26.0"
print("Installation complete. NOW RESTART THE KERNEL before running Cell 2.")

In [1]:
import os, json, random, glob
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import trl, peft, transformers
print(f"trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")
assert int(trl.__version__.split('.')[0]) >= 1, "Need TRL >= 1.0"
assert int(transformers.__version__.split('.')[0]) == 4, "transformers must be 4.x for TRL 1.5.1"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
assert torch.cuda.is_available(), "GPU not enabled — Settings → Accelerator → GPU T4"
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

2026-05-31 16:28:38.674211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780244918.895996     113 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780244918.957628     113 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780244919.470970     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780244919.471023     113 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780244919.471027     113 computation_placer.cc:177] computation placer alr

trl=1.5.1  peft=0.19.1  transformers=4.57.6
GPU: Tesla T4 | VRAM: 15.6 GB


In [2]:
spider     = load_dataset("xlangai/spider")
train_data = list(spider["train"]); dev_data = list(spider["validation"])
print(f"Train: {len(train_data)}, Dev: {len(dev_data)}")

matches = glob.glob("/kaggle/input/**/tables.json", recursive=True)
assert matches, "tables.json not found — attach your spider-tables dataset (Add Input)."
with open(matches[0]) as f:
    tables_data = json.load(f)
print(f"Loaded {len(tables_data)} schemas from {matches[0]}")

overlap = set(e["db_id"] for e in train_data) & set(e["db_id"] for e in dev_data)
print(f"Shared db_ids train/dev: {len(overlap)} (0 is correct — Spider is cross-domain)")

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Train: 7000, Dev: 1034
Loaded 166 schemas from /kaggle/input/datasets/prakashmanthan/spider-tables/tables.json
Shared db_ids train/dev: 0 (0 is correct — Spider is cross-domain)


In [3]:
def build_schema_index(tables_data):
    return {e["db_id"]: e for e in tables_data}

def get_schema_prompt(db_id, schema_index):
    if db_id not in schema_index:
        raise ValueError(f"db_id '{db_id}' not found")
    db = schema_index[db_id]
    tables, columns = db["table_names_original"], db["column_names_original"]
    col_types, pks, fks = db["column_types"], set(db["primary_keys"]), db["foreign_keys"]
    tmap = {"text":"TEXT","number":"REAL","time":"TEXT","boolean":"INTEGER","others":"TEXT"}
    parts = []
    for t_idx, tname in enumerate(tables):
        cols = []
        for c_idx, (t_id, cname) in enumerate(columns):
            if t_id != t_idx: continue
            pk = " PRIMARY KEY" if c_idx in pks else ""
            cols.append(f"  {cname} {tmap.get(col_types[c_idx].lower(),'TEXT')}{pk}")
        if cols:
            parts.append(f"CREATE TABLE {tname} (\n" + ",\n".join(cols) + "\n)")
    fk_lines = [f"-- {tables[columns[a][0]]}.{columns[a][1]} references {tables[columns[b][0]]}.{columns[b][1]}"
                for a, b in fks]
    s = "\n\n".join(parts)
    if fk_lines: s += "\n\n" + "\n".join(fk_lines)
    return s

schema_index = build_schema_index(tables_data)
print(get_schema_prompt(train_data[0]["db_id"], schema_index))

CREATE TABLE department (
  Department_ID REAL PRIMARY KEY,
  Name TEXT,
  Creation TEXT,
  Ranking REAL,
  Budget_in_Billions REAL,
  Num_Employees REAL
)

CREATE TABLE head (
  head_ID REAL PRIMARY KEY,
  name TEXT,
  born_state TEXT,
  age REAL
)

CREATE TABLE management (
  department_ID REAL PRIMARY KEY,
  head_ID REAL,
  temporary_acting TEXT
)

-- management.head_ID references head.head_ID
-- management.department_ID references department.Department_ID


In [4]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_LEN  = 1024
SYSTEM_PROMPT = ("You are an expert SQL generator. Given a database schema and a natural language "
                 "question, write the correct SQL query. Output only the SQL query with no explanation or markdown.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "right"
print(f"pad={tokenizer.pad_token!r}({tokenizer.pad_token_id}) eos={tokenizer.eos_token!r}({tokenizer.eos_token_id})")

def to_prompt_completion(ex):
    schema = get_schema_prompt(ex["db_id"], schema_index)
    user = f"### Database Schema:\n{schema}\n\n### Question:\n{ex['question']}"
    return {"prompt": [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}],
            "completion": [{"role":"assistant","content":ex["query"]}]}

kept, dropped, skipped = [], 0, 0
for ex in train_data:
    try:
        rec = to_prompt_completion(ex)
    except ValueError:
        skipped += 1; continue
    text = tokenizer.apply_chat_template(rec["prompt"]+rec["completion"], tokenize=False, add_generation_prompt=False)
    if len(tokenizer(text)["input_ids"]) <= MAX_LEN:
        kept.append(rec)
    else:
        dropped += 1
print(f"Kept {len(kept)} | dropped {dropped} (>{MAX_LEN} tokens) | skipped {skipped}")
train_dataset = Dataset.from_list(kept)
print("Columns:", train_dataset.column_names)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pad='<|endoftext|>'(151643) eos='<|im_end|>'(151645)
Kept 6836 | dropped 164 (>1024 tokens) | skipped 0
Columns: ['prompt', 'completion']


In [5]:
# Cell 6 — complete version
import os, gc
os.environ["CUDA_VISIBLE_DEVICES"] = "0"    # single GPU before model loads
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"  # reduces fragmentation

# Safety — clear any leftover GPU state
if 'model' in dir():
    del model
torch.cuda.empty_cache()
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    dtype=torch.float16, device_map="auto", trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, gradient_checkpointing_kwargs={"use_reentrant": True})
print("Base dtype:", model.dtype, "| VRAM:", round(torch.cuda.memory_allocated(0)/1e9,2), "GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base dtype: torch.float32 | VRAM: 3.02 GB


In [6]:
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
print("LoRA config ready.")

LoRA config ready.


In [7]:
sft_config = SFTConfig(
    output_dir="./qwen-text2sql-checkpoints", report_to="none",
    logging_steps=50, save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=2, gradient_accumulation_steps=8,   # OOM? -> 1 and 16
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    weight_decay=0.001, max_grad_norm=0.3,
    fp16=True, bf16=False,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": True},
    optim="paged_adamw_8bit", dataloader_pin_memory=False,
    group_by_length=True,
    max_length=MAX_LEN, completion_only_loss=True, packing=False,
    seed=SEED,
)
trainer = SFTTrainer(model=model, args=sft_config, train_dataset=train_dataset,
                     processing_class=tokenizer, peft_config=lora_config)

import collections
before = collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)
for p in trainer.model.parameters():
    if p.requires_grad: p.data = p.data.float()
after = collections.Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad)
print("Trainable dtypes BEFORE:", dict(before), "| AFTER:", dict(after))

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
assert trainable > 0
labels = trainer.data_collator([trainer.train_dataset[0]])["labels"][0]
sup = trainer.data_collator([trainer.train_dataset[0]])["input_ids"][0][labels != -100]
print(f"Trainable params: {trainable:,}")
print("Supervised text:", tokenizer.decode(sup))
print("All checks passed — run 8b.")

Tokenizing train dataset:   0%|          | 0/6836 [00:00<?, ? examples/s]

Trainable dtypes BEFORE: {'torch.bfloat16': 392} | AFTER: {'torch.float32': 392}
Trainable params: 40,370,176
Supervised text: SELECT count(*) FROM head WHERE age  >  56<|im_end|>

All checks passed — run 8b.


In [8]:
# Cell 8b — training + immediate push
trainer.train()
print("Training complete.")

# Push immediately — don't wait for Cell 9
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
REPO_ID  = "Gansaw98/qwen2.5-coder-7b-text2sql-spider"
ADAPTER  = "./qwen-text2sql-adapters"

trainer.model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print("Adapter saved locally.")

api = HfApi()
api.upload_folder(
    folder_path=ADAPTER,
    repo_id=REPO_ID,
    repo_type="model",
    token=hf_token,
)
print(f"Pushed: https://huggingface.co/{REPO_ID}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as s

Step,Training Loss
50,0.115300
100,0.090000
150,0.080600
200,0.080700
250,0.071000
300,0.069500
350,0.067200
400,0.065100
450,0.045400
500,0.032700


Training complete.
Adapter saved locally.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed: https://huggingface.co/Gansaw98/qwen2.5-coder-7b-text2sql-spider
